IDREES
# Migrant generation (built from 3 columns)
1- What is your country of origin? (compared with the country of residence)
2- In which country are you located? (if it differs from the country of origin then -> first generation)
3- Which description fits your parents’ country of origin the best? (if a non-EU parent and living in the country of origin then-> second generation)
origin != located (currently)= 1st gen
same country + non eu parents = 2nd gen 
else -> non migrant 
# ON1-ON2-ON3 cleaned and processed 
# DVP1- DVP3- DVP4 cleaned and processed 
# PF = mean(F1, F2)


In [175]:
import pandas as pd
import numpy as np

In [176]:
#(Idrees) read medeea's final combined file so our located + origin cleaning is added on top of her work (likert + parents)
df = pd.read_excel('Responses_cleaned_finalMedeea.xlsx')
df.shape

(685, 75)

Cleaning: "In which country are you located"
normalize (strip + lowercase) then map every variant to one clean country name.

In [ ]:


col = 'In which country are you located?'

In [178]:
#(Idrees)check the raw values
df[col].value_counts(dropna=False)

In which country are you located?
The Netherlands    298
Germany            151
Spain              113
Greece              76
Austria              4
United Kingdom       4
Malta                3
NaN                  3
Canada               3
France               3
Sweden               3
United States        2
Egypt                2
Belgium              2
Turkey               2
Morocco              2
Portugal             2
New Zealand          1
Singapore            1
South Africa         1
Switzerland          1
Poland               1
Italy                1
Hong Kong            1
Spiliotopoulos       1
Jordan               1
Brazil               1
Norway               1
India                1
Name: count, dtype: int64

In [179]:
#(Idrees)number of unique values and missing count
print('Unique values:', df[col].nunique(dropna=True))
print('Missing:', df[col].isna().sum())

Unique values: 28
Missing: 3


In [180]:
#(Idrees)- fix all netherlands variants to "The Netherlands"
netherlands_variants = [
    'netherlands', 'the netherlands', 'nederland', 'niederlande', 'holland', 'nl',
    'netherland', 'the netherlans', 'netherworld',
    'amsterdam', 'amsterxam', 'uithoorn', 'netherlands, amsterdam',
]

#compare on a stripped + lowercased version so spacing and case don't matter
normalized = df[col].astype(str).str.strip().str.lower()
df.loc[normalized.isin(netherlands_variants), col] = 'The Netherlands'

#check the result
df[col].value_counts(dropna=False)
print(normalized.isin(netherlands_variants).sum(), 'rows updated to "The Netherlands"')

298 rows updated to "The Netherlands"


In [181]:
#(Idrees)-clean all the other countries the same way, using a normalized then clean-name map.
#keys are stripped + lowercased, so case/spacing variants are handled automatically.
country_map = {
    'germany': 'Germany', 'deutschland': 'Germany', 'german': 'Germany',
    'spain': 'Spain', 'españa': 'Spain', 'espana': 'Spain',
    'greece': 'Greece', 'greece.': 'Greece',
    'turkiye': 'Turkey', 'türkiye': 'Turkey', 'تركيا': 'Turkey',
    'egypt': 'Egypt', 'cairo': 'Egypt',
    'united kingdom': 'United Kingdom', 'uk': 'United Kingdom', 'england': 'United Kingdom',
    'usa': 'United States', 'united states': 'United States',
    'malta': 'Malta', 'france': 'France', 'sweden': 'Sweden', 'austria': 'Austria',
    'morocco': 'Morocco', 'belgium': 'Belgium', 'canada': 'Canada', 'portugal': 'Portugal',
    'brazil': 'Brazil', 'jordan': 'Jordan', 'norway': 'Norway', 'india': 'India',
    'new zealand': 'New Zealand', 'poland': 'Poland', 'south africa': 'South Africa',
    'switzerland': 'Switzerland', 'italy': 'Italy', 'hong kong': 'Hong Kong',
    'singapore': 'Singapore',
}

normalized = df[col].astype(str).str.strip().str.lower()
mapped = normalized.map(country_map)
#where we have a mapping use it; otherwise keep the current value 
df[col] = mapped.fillna(df[col])

#check the result
df[col].value_counts(dropna=False)

In which country are you located?
The Netherlands    298
Germany            151
Spain              113
Greece              76
Austria              4
United Kingdom       4
Malta                3
NaN                  3
Canada               3
France               3
Sweden               3
United States        2
Egypt                2
Belgium              2
Turkey               2
Morocco              2
Portugal             2
New Zealand          1
Singapore            1
South Africa         1
Switzerland          1
Poland               1
Italy                1
Hong Kong            1
Spiliotopoulos       1
Jordan               1
Brazil               1
Norway               1
India                1
Name: count, dtype: int64

Cleaning: "What is your country of origin?"
Same approach as the previous column.

Key idea: **origin = the person's OWN country** (where they were born / live). Heritage/ancestry is NOT origin — it is already captured by the parents column and used separately in the MG1 logic. Putting heritage into origin would count it twice and wrongly turn 2nd-generation people into 1st-generation.

Judgment-call defaults:
- **Heritage answers** (born/living in country X but family from Y, e.g. "I was born in Germany, but my family is from Sri Lanka", or answers that only mention the parents) -> origin = the country where they live. The parents column then decides 2nd-gen vs non-migrant.
- **Actual migrants** (they name a foreign country they came from and now live elsewhere, e.g. "England/Spain" living in NL) -> origin = that foreign country, so origin ≠ located -> 1st generation.
- **Both-EU multi-country** (e.g. "Netherlands/Greece") -> first-listed country.
- Junk / uninterpretable answers -> Unknown (NaN)

In [ ]:
#(Idrees) the second column we are cleaning
col_o = 'What is your country of origin?'


_raw_origin = pd.read_excel('Responses.xlsx')[['ID', col_o]].rename(columns={col_o: '_raw_origin'})
df = df.drop(columns=['_raw_origin'], errors='ignore').merge(_raw_origin, on='ID', how='left')
df[col_o] = df['_raw_origin']
df = df.drop(columns='_raw_origin')

#check the raw values + missing
print('Unique values:', df[col_o].nunique(dropna=True))
print('Missing:', df[col_o].isna().sum())
df[col_o].value_counts(dropna=False)

In [ ]:
#(Idrees) map every variant (case/spacing/typo/language/adjective) to one clean country name
#
#DONE (was TODO): handling people who wrote TWO/MORE countries or a heritage statement.
#KEY POINT: "origin" = the person's OWN country (where they were born / live). Heritage/ancestry is
#NOT origin - it is already captured by the parents column and used separately in the MG1 logic below.
#If we put heritage into origin we count it twice and wrongly turn 2nd-gen people into 1st-gen.
#So:
#  - Heritage answers (born/living in country X but family from Y, or only mentioning parents)
#      -> origin = the country where they live (located). The parents column then decides 2nd-gen vs non-migrant.
#  - People who actually migrated (name a foreign country they came from, now living elsewhere)
#      -> origin = that foreign country, so origin != located -> 1st generation.
origin_map = {
    #cities/region + adjective + typos
    'netherlands':'The Netherlands','the netherlands':'The Netherlands','nederland':'The Netherlands',
    'nederlands':'The Netherlands','netherland':'The Netherlands','nehterlands':'The Netherlands',
    'holland':'The Netherlands','nl':'The Netherlands','amsterdam':'The Netherlands',
    'fryslan':'The Netherlands','dutch':'The Netherlands',

    'germany':'Germany','deutschland':'Germany','german':'Germany','getmany':'Germany','germsny':'Germany',

    'spain':'Spain','españa':'Spain','espana':'Spain',

    'greece':'Greece','greece.':'Greece','greek':'Greece','creta island in greece':'Greece',

    'syria':'Syria','syrien':'Syria','سورية':'Syria',

    'morocco':'Morocco','marocco':'Morocco','morrocco':'Morocco','morroco':'Morocco',

    'turkiye':'Turkey','turkey':'Turkey','egypt':'Egypt',
    'united kingdom':'United Kingdom','uk':'United Kingdom','england':'United Kingdom',
    'usa':'United States','united states':'United States','palestine':'Palestine',

    'suriname':'Suriname','surinam':'Suriname','china':'China','chinese':'China',
    'bulgarian':'Bulgaria','irland':'Ireland','kazahstan':'Kazakhstan',
    'france':'France','frankrijk':'France',

    'india':'India','lebanon':'Lebanon','ghana':'Ghana','italy':'Italy','pakistan':'Pakistan',
    'portugal':'Portugal','eritrea':'Eritrea','vietnam':'Vietnam','taiwan':'Taiwan','canada':'Canada',
    'romania':'Romania','colombia':'Colombia','brazil':'Brazil','indonesia':'Indonesia','ecuador':'Ecuador',
    'nigeria':'Nigeria','moldova':'Moldova','aruba':'Aruba','peru':'Peru','iran':'Iran',
    'philippines':'Philippines','iraq':'Iraq','nepal':'Nepal','russia':'Russia','austria':'Austria',
    'luxembourg':'Luxembourg','argentina':'Argentina','south africa':'South Africa','latvia':'Latvia',
    'ukraine':'Ukraine','venezuela':'Venezuela','jamaica':'Jamaica','lithuania':'Lithuania','sri lanka':'Sri Lanka',

    #multi-country answers where the person names their OWN country of origin (not heritage):
    'england/ spain':'United Kingdom',     #lives in NL, came from the UK -> origin UK, 1st gen
    'netherlands/ spain':'The Netherlands',#lives in NL, both parents EU -> origin NL
    'netherlands/greece':'The Netherlands',#lives in NL, both parents EU -> origin NL
    'born in syria but originally im palestinian.':'Syria',  #born in Syria, migrated to NL -> origin Syria, 1st gen
    'suriname, molukken, irak':'The Netherlands',            #lives in NL, both parents born/brought up in NL -> origin NL
}

#heritage answers: the person was born/lives in their country of residence but has foreign family roots
#(or only describes their parents). Heritage belongs in the parents column, NOT origin -> origin = located,
#so the parents column alone decides 2nd-gen vs non-migrant.
heritage_answers = [
    'netherlands, ethnically from lebanon',                              #lives NL, family Lebanon
    'ntherlands/surinam',                                               #lives NL, family Suriname
    'dutch,surinamese & indonesian',                                    #lives NL, family Surinamese/Indonesian
    'egypt/germany',                                                    #lives Germany, Egyptian parent
    'i was born in germany, but my family is originally from sri lanka.',#born+lives Germany, family Sri Lanka
    'my parents are from pakistan',                                     #lives NL, only states parents (Pakistan)
]

normalized = df[col_o].astype(str).str.strip().str.lower()
#where we have a mapping use it; otherwise keep the current value
df[col_o] = normalized.map(origin_map).fillna(df[col_o])

#heritage answers -> origin = country of residence (df[col] = cleaned "located"); parents column decides generation
heritage_rows = normalized.isin(heritage_answers)
df.loc[heritage_rows, col_o] = df.loc[heritage_rows, col]

#junk / uninterpretable free-text -> Unknown (NaN)
junk = ['yes', 'manerf']
df.loc[df[col_o].astype(str).str.strip().str.lower().isin(junk), col_o] = np.nan

#check the result
df[col_o].value_counts(dropna=False)

#(Idrees) build the single migrant-generation variable (MG1) from the 3 cleaned columns 
-First generation: country of origin differs from country of residence (the person migrated)
-Second generation: same country, but at least one parent is from outside the EU (the parents migrated)
-Non-migrant: same country and both parents from the EU
-Unknown: country of origin or residence is missing

In [184]:

origin  = df[col_o]   #cleaned country of origin
located = df[col]     #cleaned country of residence

# "has at least one non-EU parent", from Medeea's one-hot parents columns
nonEU_parent_cols = [
    'origin_Both parents are from outside the EU',
    'origin_One parent is from the EU and one parent is outside the EU',
]
present = [c for c in nonEU_parent_cols if c in df.columns]
nonEU_parent = df[present].astype(bool).any(axis=1) if present else pd.Series(False, index=df.index)

same    = origin.eq(located)
missing = origin.isna() | located.isna()

#MG1 = migrant generation
df['MG1'] = np.select(
    [missing, ~same, same & nonEU_parent],
    ['Unknown', 'First generation', 'Second generation'],
    default='Non-migrant',
)

df['MG1'].value_counts(dropna=False)

MG1
Non-migrant          384
First generation     225
Second generation     71
Unknown                5
Name: count, dtype: int64

## ON1,ON2,ON3

In [185]:
#(Idrees) map Likert text to 1-5, then reverse-code ON2
likert = {'Strongly disagree':1,'Disagree':2,'Neutral':3,'Agree':4,'Strongly agree':5}

on_cols = {
    'ON1': 'I feel like my feed shows me a good mix of topics to stay informed (e.g., news, social issues, or different opinons)',
    'ON2': 'I believe that the algorithm hides certain social or political views',
    'ON3': 'I feel like I have a fair chance to be seen/heard',
}
for code, c in on_cols.items():
    df[code] = pd.to_numeric(df[c].replace(likert), errors='coerce')

df['ON2'] = 6 - df['ON2']   #reverse ON2

print(df[['ON1','ON2','ON3']].describe())


print(df[['ON1','ON2','ON3']].corr().round(2))

              ON1         ON2         ON3
count  676.000000  676.000000  673.000000
mean     3.337278    2.318047    3.083210
std      0.973045    0.906454    0.812239
min      1.000000    1.000000    1.000000
25%      3.000000    2.000000    3.000000
50%      4.000000    2.000000    3.000000
75%      4.000000    3.000000    4.000000
max      5.000000    5.000000    5.000000
      ON1   ON2   ON3
ON1  1.00  0.04  0.14
ON2  0.04  1.00  0.16
ON3  0.14  0.16  1.00


Why we reverse ON2? 
think of it like a thumbs up score
ON1: “My feed keeps me informed” -> 5 = thumbs up
ON3: “I get a fair chance to be seen” -> 5 = thumbs up 
ON2: “The algorithm hides views” -> 5 = thumbs down

ON2 is the odd one out, a high score there means something bad.
so we flip it so that all 3 

## DVP1,DVP3,DVP4

In [186]:
#(Idrees) text -> 1-5, reverse DVP3 and DVP4
dvp_cols = {
    'DVP1': 'When I post about topics like politics or social issues, it feels like my voice can be seen and heard',
    'DVP3': 'I feel like the algorithm shapes what political topics I get to see',
    'DVP4': 'I rarely see posts that challenge my own political views unless I search for them',
}
for code, c in dvp_cols.items():
    df[code] = pd.to_numeric(df[c].replace(likert), errors='coerce')

df['DVP3'] = 6 - df['DVP3']
df['DVP4'] = 6 - df['DVP4']


print(df[['DVP1','DVP3','DVP4']].corr().round(2))

      DVP1  DVP3  DVP4
DVP1  1.00 -0.05 -0.14
DVP3 -0.05  1.00  0.23
DVP4 -0.14  0.23  1.00


In [187]:
#(Idrees) frequency checking: any item where almost everyone gives the same answer?
for c in ['ON1','ON2','ON3','DVP1','DVP3','DVP4']:
    print(c, df[c].value_counts(normalize=True).sort_index().round(2).to_dict())

ON1 {1.0: 0.03, 2.0: 0.18, 3.0: 0.28, 4.0: 0.42, 5.0: 0.08}
ON2 {1.0: 0.17, 2.0: 0.46, 3.0: 0.26, 4.0: 0.09, 5.0: 0.01}
ON3 {1.0: 0.02, 2.0: 0.19, 3.0: 0.48, 4.0: 0.27, 5.0: 0.03}
DVP1 {1.0: 0.06, 2.0: 0.17, 3.0: 0.47, 4.0: 0.27, 5.0: 0.03}
DVP3 {1.0: 0.22, 2.0: 0.51, 3.0: 0.2, 4.0: 0.05, 5.0: 0.01}
DVP4 {1.0: 0.09, 2.0: 0.36, 3.0: 0.3, 4.0: 0.23, 5.0: 0.03}


# PF  (perceived fairness)
-F1: "I trust the algorithm to recommend me content fairly" (trust in algorithmic fairness)
-F2: "I think social media is fair2" (general fairness, end of survey)

PF = the mean of F1 and F2, which keeps it on the original 1–5 scale (comparable to the ON/DVP items). We also correlate F1 & F2 to check they move together before averaging.

In [ ]:
#(Idrees)- 
f_cols = {
    'F1': 'I trust the algorithm to recommend me content fairly',
    'F2': 'I think social media is fair2',
}
for code, c in f_cols.items():
    df[code] = pd.to_numeric(df[c].replace(likert), errors='coerce')


df['PF'] = df[['F1','F2']].mean(axis=1)

print('PF mean (overall):', round(df['PF'].mean(), 2))
print(df[['F1','F2','PF']].describe().round(2))

print('\nF1-F2 correlation:')
print(df[['F1','F2']].corr().round(2))

PF mean (overall): 2.76
           F1      F2      PF
count  675.00  676.00  679.00
mean     2.85    2.66    2.76
std      0.94    0.92    0.80
min      1.00    1.00    1.00
25%      2.00    2.00    2.00
50%      3.00    3.00    3.00
75%      4.00    3.00    3.50
max      5.00    5.00    5.00

F1-F2 correlation:
      F1    F2
F1  1.00  0.48
F2  0.48  1.00


In [189]:
#save to Medeea's final combined file
df.to_excel('Responses_cleaned_finalMedeea.xlsx', index=False)
print('saved in Responses_cleaned_finalMedeea.xlsx')

saved in Responses_cleaned_finalMedeea.xlsx
